# Entregable II — Reducción de dimensión: PCA y UMAP (20 %)

**Proyecto:** Predicción de la intención de compra de usuarios en línea mediante aprendizaje supervisado.
**Curso:** Modelos y Simulación de Sistemas II — Universidad de Antioquia.

Este notebook reproduce la comparación de PCA y UMAP con la línea base de Random Forest
y la MLP. La reducción se ajusta dentro del `Pipeline`, solo con el 60 % de entrenamiento,
y se evalúa en validación y prueba.

> **Nota de tiempo:** `RECALCULAR = True` reajusta PCA y UMAP y puede tardar varios
> minutos. Con `False` se leen las tablas y figuras guardadas en `resultados_corregidos/`.

In [ ]:
import os
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))
RES = RAIZ / "resultados_corregidos"
FIG = RAIZ / "informe_entregable_II_latex" / "figuras"

import json
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("Directorio de trabajo:", RAIZ)

RECALCULAR = False  # True para reajustar PCA/UMAP desde el CSV

if RECALCULAR:
    import reduccion
    import figuras_reduccion
    reduccion.main(resultados=RES)
    figuras_reduccion.generar(resultados=RES, figuras=FIG)
else:
    print("Se cargan los resultados guardados. Cambie RECALCULAR = True para reajustar.")

## 1. Línea base

Random Forest y la MLP con los hiperparámetros ganadores de la sección del 30 %.

In [ ]:
base = pd.read_csv("resultados_corregidos/metricas_particiones.csv")
base[(base.particion == "test") & (base.modelo.isin(["RandomForest", "MLP"]))][
    ["modelo", "AP", "ROC_AUC", "F1", "precision", "recall"]
].round(4)

## 2. PCA: varianza acumulada

PCA retiene 28 componentes para el 95 % de la varianza: 61,64 % menos columnas que las
73 originales. El primer componente concentra 31,09 % de la varianza.

In [ ]:
var = pd.read_csv("resultados_corregidos/pca_varianza.csv")
var.tail(8).round(4)

In [ ]:
display(Image(filename="informe_entregable_II_latex/figuras/fig11_pca_varianza.png"))

## 3. Comparación PCA, UMAP y línea base

La tabla `reduccion_comparacion.csv` contiene AP, ROC-AUC, F1, reducción porcentual y
las diferencias frente a la base para PCA y UMAP de 10 y 2 componentes. La variante
de 2 componentes también se entrena y evalúa con ambos clasificadores.

In [ ]:
comp = pd.read_csv("resultados_corregidos/reduccion_comparacion.csv")
comp[comp.particion == "test"][
    ["modelo", "AP", "ROC_AUC", "F1", "reduccion_pct",
     "AP_base", "delta_AP", "delta_ROC_AUC", "delta_F1", "tiempo_seg"]
].round(4)

In [ ]:
u2 = pd.read_csv("resultados_corregidos/reduccion_umap2.csv")
u2[u2.particion == "test"][
    ["modelo", "AP", "ROC_AUC", "F1", "tiempo_seg"]
].round(4)

## 4. Figuras de la reducción

(a) Varianza acumulada de PCA y (b) AP de prueba con cada representación. La proyección
UMAP 2D se muestra aparte.

In [ ]:
display(Image(filename="informe_entregable_II_latex/figuras/fig13_comparacion_reduccion.png"))
display(Image(filename="informe_entregable_II_latex/figuras/fig12_umap_2d.png"))

## 5. Reproducción desde el CSV

Si `RECALCULAR = True`, se reajustan PCA y UMAP con la misma partición y semilla del
informe. Los resultados se sobrescriben en `resultados_corregidos/`.

In [ ]:
print("Recalcular desde el CSV:", RECALCULAR)

## 6. Lectura de resultados

- PCA conserva el AP dentro de 1,7 puntos (RF 0,7352; MLP 0,7211) con 61,64 % menos
  columnas; el ROC-AUC se mantiene por encima de 0,92.
- UMAP de 10 componentes reduce 86,30 % de las columnas pero baja el AP a 0,6650 (RF) y
  0,6236 (MLP); con 2 componentes cae aún más.
- Para estas configuraciones, partición y semilla, las variantes UMAP obtienen menor AP
  puntual que la base; esto no establece la causa de la diferencia.